# 🏗️ InFoundry Architect - Simple Training

No Oumi dependencies - uses transformers + peft directly.

In [ ]:
# Step 1: Install only what we need
!pip install -q transformers peft accelerate bitsandbytes datasets trl
print('✅ Installed')

In [ ]:
# Check GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Step 2: Upload training data
from google.colab import files
print('Upload generated_training_data.jsonl:')
uploaded = files.upload()

In [ ]:
# Step 3: Load and prepare data
import json
from datasets import Dataset

# Load JSONL
with open('generated_training_data.jsonl') as f:
    examples = [json.loads(line) for line in f]

# Convert to text format for SFT
def format_example(ex):
    messages = ex['messages']
    text = f"<|im_start|>system\n{messages[0]['content']}<|im_end|>\n"
    text += f"<|im_start|>user\n{messages[1]['content']}<|im_end|>\n"
    text += f"<|im_start|>assistant\n{messages[2]['content']}<|im_end|>"
    return {'text': text}

dataset = Dataset.from_list([format_example(ex) for ex in examples])
print(f'✅ Loaded {len(dataset)} examples')
print(dataset[0]['text'][:500])

In [ ]:
# Step 4: Load model with LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load model in 4-bit to save memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

# Prepare for training
model = prepare_model_for_kbit_training(model)

# Add LoRA
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ Model loaded with LoRA')

In [ ]:
# Step 5: Train with TRL SFTTrainer
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./trained_model',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    max_steps=500,
    logging_steps=25,
    save_steps=100,
    warmup_ratio=0.1,
    fp16=True,
    max_seq_length=1024,
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
)

print('🚀 Starting training...')
trainer.train()
print('✅ Training complete!')

# Save
trainer.save_model('./trained_model')
tokenizer.save_pretrained('./trained_model')

In [ ]:
# Step 6: Test the model
from peft import PeftModel

# Reload for inference
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, './trained_model')

# Test
test_input = 'Services: [api, auth], Language: python, DB: postgres, Cloud: aws, Latency p95: 100ms, Cost: $50/day'

system = '''You are an expert cloud architect. Respond with ONLY valid JSON:
{"architecture": {"pattern": "...", "components": [...], "topology": "...", "scaling_strategy": "...", "estimated_cost_tier": "...", "rationale": "..."}, "inputs": {"service_count": N, "cloud_provider": "..."}, "source": "ai_recommendation"}'''

prompt = f'<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\n{test_input}<|im_end|>\n<|im_start|>assistant\n'

inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=400, temperature=0.1, do_sample=True)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print('Input:', test_input)
print('\nOutput:', result.split('assistant')[-1].strip())

In [ ]:
# Step 7: Download trained model
!zip -r trained_model.zip ./trained_model
from google.colab import files
files.download('trained_model.zip')
print('✅ Downloaded!')